# Library 剩餘24段｜先 dry-run，等待確認
Rosie 13 / Princess 11。已接受六句只原樣複製，不生成。Bakery及Park不在輸入包。免費Colab GPU可能有時數及資源限制，無付費API。

In [ ]:
import sys, subprocess, os
os.environ['HF_HUB_DISABLE_IMPLICIT_TOKEN']='1'
os.environ['HF_HUB_DISABLE_TELEMETRY']='1'
subprocess.check_call(['apt-get','-qq','update'])
subprocess.check_call(['apt-get','-qq','install','-y','espeak-ng','sox','libsndfile1'])
subprocess.check_call([sys.executable,'-m','pip','install','-q','qwen-tts==0.1.1','transformers==4.57.3','accelerate==1.12.0','kokoro==0.9.4','soundfile','ipywidgets>=8,<9','huggingface-hub>=0.34,<1'])
print('套件安裝完成。若Colab提示套件已載入，請重新啟動執行階段，再從上傳格繼續。')

## 2 上傳 library-remaining24-inputs.zip

In [ ]:
from google.colab import files
from pathlib import Path
import sys,uuid,zipfile,io
up=files.upload()
raw=next(iter(up.values()))
WORK=Path('/content/library24_'+uuid.uuid4().hex[:8])
with zipfile.ZipFile(io.BytesIO(raw)) as z:
 for name in z.namelist():
  p=(WORK/name).resolve()
  if not p.is_relative_to(WORK.resolve()):raise ValueError('Unsafe ZIP')
 z.extractall(WORK)
sys.path.insert(0,str(WORK))
import remaining_runner as runner
OUTPUT=Path('/content/library30_'+uuid.uuid4().hex[:8])

## 3 Dry-run｜執行後先停，把清單回傳確認

In [ ]:
from IPython.display import display,Markdown
plan=runner.generate(WORK,OUTPUT,run=False)
display(Markdown('|角色|ID|英文原文|\n|---|---|---|\n'+'\n'.join('|'+a['role']+'|'+a['lineId']+'|'+a['text']+'|' for a in plan['jobs'])))
print('24 jobs / 0 generated. Protected six hashes verified.')

## 4 等家長確認24段清單後才啟用

In [ ]:
RUN_GENERATION=False
CONFIRMATION='' # 確認後填 CONFIRM_LIBRARY_REMAINING_24
if RUN_GENERATION:
 result=runner.generate(WORK,OUTPUT,run=True,confirmation=CONFIRMATION)
else:print('未生成，等待清單確認。')

## 5｜30句逐句驗收（只讀現有 OUTPUT，不重新生成）
選擇與備註自動保存；六句已接受鎖定。可匯出/載入JSON及下載完整30句ZIP。重新執行本區塊會載入已保存紀錄。

In [ ]:
# Library 30句人工驗收；只讀WAV/manifest，不載入或呼叫TTS模型。
from pathlib import Path
import json, hashlib, datetime, zipfile, io, html, uuid
import numpy as np
import soundfile as sf

EXPECTED = {'rosie-511b9ae26efed5a8': {'assetId': 'rosie-511b9ae26efed5a8', 'role': 'rosie', 'voiceId': 'qwen_rosie_r2', 'text': 'Welcome to the Halloween Library!', 'filename': 'rosie__rosie-511b9ae26efed5a8__qwen_rosie_r2.wav', 'src': 'audio/rosie__rosie-511b9ae26efed5a8__qwen_rosie_r2.wav', 'lineIds': ['hw.library.welcome.lines.01']}, 'rosie-90eb7bf71f773ef6': {'assetId': 'rosie-90eb7bf71f773ef6', 'role': 'rosie', 'voiceId': 'qwen_rosie_r2', 'text': 'A crown. Crown.', 'filename': 'rosie__rosie-90eb7bf71f773ef6__qwen_rosie_r2.wav', 'src': 'audio/rosie__rosie-90eb7bf71f773ef6__qwen_rosie_r2.wav', 'lineIds': ['hw.library.teach-crown.lines.01']}, 'rosie-87319e5d959e19f2': {'assetId': 'rosie-87319e5d959e19f2', 'role': 'rosie', 'voiceId': 'qwen_rosie_r2', 'text': 'A crown. A pink dress. Who is it?', 'filename': 'rosie__rosie-87319e5d959e19f2__qwen_rosie_r2.wav', 'src': 'audio/rosie__rosie-87319e5d959e19f2__qwen_rosie_r2.wav', 'lineIds': ['hw.library.find-princess.lines.01']}, 'rosie-c190ddf93ee3874a': {'assetId': 'rosie-c190ddf93ee3874a', 'role': 'rosie', 'voiceId': 'qwen_rosie_r2', 'text': 'Crown. Pink dress.', 'filename': 'rosie__rosie-c190ddf93ee3874a__qwen_rosie_r2.wav', 'src': 'audio/rosie__rosie-c190ddf93ee3874a__qwen_rosie_r2.wav', 'lineIds': ['hw.library.find-princess.hint.01']}, 'rosie-7affab230c87e815': {'assetId': 'rosie-7affab230c87e815', 'role': 'rosie', 'voiceId': 'qwen_rosie_r2', 'text': 'Good try. Look for the pink dress.', 'filename': 'rosie__rosie-7affab230c87e815__qwen_rosie_r2.wav', 'src': 'audio/rosie__rosie-7affab230c87e815__qwen_rosie_r2.wav', 'lineIds': ['hw.library.find-princess.retry.01']}, 'rosie-ec2ecfd3d2ddfdeb': {'assetId': 'rosie-ec2ecfd3d2ddfdeb', 'role': 'rosie', 'voiceId': 'qwen_rosie_r2', 'text': 'Yes! She is the princess! The princess has a crown!', 'filename': 'rosie__rosie-ec2ecfd3d2ddfdeb__qwen_rosie_r2.wav', 'src': 'audio/rosie__rosie-ec2ecfd3d2ddfdeb__qwen_rosie_r2.wav', 'lineIds': ['hw.library.crown-on.lines.01']}, 'princess-3720e4220c8a5ce4': {'assetId': 'princess-3720e4220c8a5ce4', 'role': 'princess', 'voiceId': 'af_bella', 'text': 'Hello, Stephanie!', 'filename': 'princess__princess-3720e4220c8a5ce4__af_bella.wav', 'src': 'audio/princess__princess-3720e4220c8a5ce4__af_bella.wav', 'lineIds': ['hw.library.hello.lines.01']}, 'princess-e728f8e179e85c02': {'assetId': 'princess-e728f8e179e85c02', 'role': 'princess', 'voiceId': 'af_bella', 'text': 'Where is my book? My book has a crown.', 'filename': 'princess__princess-e728f8e179e85c02__af_bella.wav', 'src': 'audio/princess__princess-e728f8e179e85c02__af_bella.wav', 'lineIds': ['hw.library.find-book.lines.01']}, 'rosie-68054a70c21c65b6': {'assetId': 'rosie-68054a70c21c65b6', 'role': 'rosie', 'voiceId': 'qwen_rosie_r2', 'text': 'Find the crown book. Which one?', 'filename': 'rosie__rosie-68054a70c21c65b6__qwen_rosie_r2.wav', 'src': 'audio/rosie__rosie-68054a70c21c65b6__qwen_rosie_r2.wav', 'lineIds': ['hw.library.find-book.lines.02']}, 'rosie-d1b47e593b3a21df': {'assetId': 'rosie-d1b47e593b3a21df', 'role': 'rosie', 'voiceId': 'qwen_rosie_r2', 'text': 'Crown. Find the crown book.', 'filename': 'rosie__rosie-d1b47e593b3a21df__qwen_rosie_r2.wav', 'src': 'audio/rosie__rosie-d1b47e593b3a21df__qwen_rosie_r2.wav', 'lineIds': ['hw.library.find-book.hint.01']}, 'rosie-b533d7376f1b7f96': {'assetId': 'rosie-b533d7376f1b7f96', 'role': 'rosie', 'voiceId': 'qwen_rosie_r2', 'text': 'Good try. Look for the crown.', 'filename': 'rosie__rosie-b533d7376f1b7f96__qwen_rosie_r2.wav', 'src': 'audio/rosie__rosie-b533d7376f1b7f96__qwen_rosie_r2.wav', 'lineIds': ['hw.library.find-book.retry.01']}, 'rosie-6bfa133bd1d20605': {'assetId': 'rosie-6bfa133bd1d20605', 'role': 'rosie', 'voiceId': 'qwen_rosie_r2', 'text': 'Yes! The crown book!', 'filename': 'rosie__rosie-6bfa133bd1d20605__qwen_rosie_r2.wav', 'src': 'audio/rosie__rosie-6bfa133bd1d20605__qwen_rosie_r2.wav', 'lineIds': ['hw.library.book-found.lines.01']}, 'princess-571bf50fb946e5c5': {'assetId': 'princess-571bf50fb946e5c5', 'role': 'princess', 'voiceId': 'af_bella', 'text': 'I found my book! Thank you, Stephanie!', 'filename': 'princess__princess-571bf50fb946e5c5__af_bella.wav', 'src': 'audio/princess__princess-571bf50fb946e5c5__af_bella.wav', 'lineIds': ['hw.library.book-found.lines.02']}, 'princess-666130f81a2643cb': {'assetId': 'princess-666130f81a2643cb', 'role': 'princess', 'voiceId': 'af_bella', 'text': 'Let’s open the book!', 'filename': 'princess__princess-666130f81a2643cb__af_bella.wav', 'src': 'audio/princess__princess-666130f81a2643cb__af_bella.wav', 'lineIds': ['hw.library.open-book.lines.01']}, 'princess-00ce7ab0e0fa0e49': {'assetId': 'princess-00ce7ab0e0fa0e49', 'role': 'princess', 'voiceId': 'af_bella', 'text': 'This is a magic book! Let’s play Stephanie Says!', 'filename': 'princess__princess-00ce7ab0e0fa0e49__af_bella.wav', 'src': 'audio/princess__princess-00ce7ab0e0fa0e49__af_bella.wav', 'lineIds': ['hw.library.stephanie-says.lines.01']}, 'princess-81eaaa24e2930533': {'assetId': 'princess-81eaaa24e2930533', 'role': 'princess', 'voiceId': 'af_bella', 'text': 'You’re the teacher, Stephanie. I’ll follow you!', 'filename': 'princess__princess-81eaaa24e2930533__af_bella.wav', 'src': 'audio/princess__princess-81eaaa24e2930533__af_bella.wav', 'lineIds': ['hw.library.stephanie-says.lines.02']}, 'princess-f62fe81d9ea55595': {'assetId': 'princess-f62fe81d9ea55595', 'role': 'princess', 'voiceId': 'af_bella', 'text': 'I’m ready, Teacher Stephanie!', 'filename': 'princess__princess-f62fe81d9ea55595__af_bella.wav', 'src': 'audio/princess__princess-f62fe81d9ea55595__af_bella.wav', 'lineIds': ['hw.library.command-1.lines.01']}, 'princess-f4203d98fce906d9': {'assetId': 'princess-f4203d98fce906d9', 'role': 'princess', 'voiceId': 'af_bella', 'text': 'Turn around, wave your hands, climb, or rollerblade?', 'filename': 'princess__princess-f4203d98fce906d9__af_bella.wav', 'src': 'audio/princess__princess-f4203d98fce906d9__af_bella.wav', 'lineIds': ['hw.library.command-1.lines.02', 'hw.library.command-2.lines.02']}, 'rosie-9c39282e9c49070c': {'assetId': 'rosie-9c39282e9c49070c', 'role': 'rosie', 'voiceId': 'qwen_rosie_r2', 'text': 'Choose one. Tell me: “Stephanie says…”', 'filename': 'rosie__rosie-9c39282e9c49070c__qwen_rosie_r2.wav', 'src': 'audio/rosie__rosie-9c39282e9c49070c__qwen_rosie_r2.wav', 'lineIds': ['hw.library.command-1.lines.03', 'hw.library.command-2.lines.03']}, 'rosie-2518646d9595742a': {'assetId': 'rosie-2518646d9595742a', 'role': 'rosie', 'voiceId': 'qwen_rosie_r2', 'text': 'Say: “Stephanie says, wave your hands!”', 'filename': 'rosie__rosie-2518646d9595742a__qwen_rosie_r2.wav', 'src': 'audio/rosie__rosie-2518646d9595742a__qwen_rosie_r2.wav', 'lineIds': ['hw.library.command-1.hint.01', 'hw.library.command-2.hint.01']}, 'princess-02986f8d4e57964b': {'assetId': 'princess-02986f8d4e57964b', 'role': 'princess', 'voiceId': 'af_bella', 'text': 'Stephanie says… I’ll do it!', 'filename': 'princess__princess-02986f8d4e57964b__af_bella.wav', 'src': 'audio/princess__princess-02986f8d4e57964b__af_bella.wav', 'lineIds': ['hw.library.move-1.lines.01', 'hw.library.move-2.lines.01']}, 'princess-f49de3f6f81d17d7': {'assetId': 'princess-f49de3f6f81d17d7', 'role': 'princess', 'voiceId': 'af_bella', 'text': 'Your turn again, Stephanie! What should I do?', 'filename': 'princess__princess-f49de3f6f81d17d7__af_bella.wav', 'src': 'audio/princess__princess-f49de3f6f81d17d7__af_bella.wav', 'lineIds': ['hw.library.command-2.lines.01']}, 'rosie-4682559e0c393296': {'assetId': 'rosie-4682559e0c393296', 'role': 'rosie', 'voiceId': 'qwen_rosie_r2', 'text': 'You’re a great teacher, Stephanie!', 'filename': 'rosie__rosie-4682559e0c393296__qwen_rosie_r2.wav', 'src': 'audio/rosie__rosie-4682559e0c393296__qwen_rosie_r2.wav', 'lineIds': ['hw.library.pages.lines.01']}, 'princess-9df3ec60229e839e': {'assetId': 'princess-9df3ec60229e839e', 'role': 'princess', 'voiceId': 'af_bella', 'text': 'I loved playing Stephanie Says with you!', 'filename': 'princess__princess-9df3ec60229e839e__af_bella.wav', 'src': 'audio/princess__princess-9df3ec60229e839e__af_bella.wav', 'lineIds': ['hw.library.pages.lines.02']}, 'rosie-f4f915314162a16c': {'assetId': 'rosie-f4f915314162a16c', 'role': 'rosie', 'voiceId': 'qwen_rosie_r2', 'text': 'Great job, Stephanie!', 'filename': 'rosie__rosie-f4f915314162a16c__qwen_rosie_r2.wav', 'src': 'audio/rosie__rosie-f4f915314162a16c__qwen_rosie_r2.wav', 'lineIds': ['hw.library.close-book.lines.01']}, 'princess-439256ed3b9e572a': {'assetId': 'princess-439256ed3b9e572a', 'role': 'princess', 'voiceId': 'af_bella', 'text': 'I’m happy!', 'filename': 'princess__princess-439256ed3b9e572a__af_bella.wav', 'src': 'audio/princess__princess-439256ed3b9e572a__af_bella.wav', 'lineIds': ['hw.library.happy-question.lines.01']}, 'rosie-440a07b589744f63': {'assetId': 'rosie-440a07b589744f63', 'role': 'rosie', 'voiceId': 'qwen_rosie_r2', 'text': 'Are you happy?', 'filename': 'rosie__rosie-440a07b589744f63__qwen_rosie_r2.wav', 'src': 'audio/rosie__rosie-440a07b589744f63__qwen_rosie_r2.wav', 'lineIds': ['hw.library.happy-question.lines.02']}, 'princess-cde0b237677f051f': {'assetId': 'princess-cde0b237677f051f', 'role': 'princess', 'voiceId': 'af_bella', 'text': 'Let’s dance!', 'filename': 'princess__princess-cde0b237677f051f__af_bella.wav', 'src': 'audio/princess__princess-cde0b237677f051f__af_bella.wav', 'lineIds': ['hw.library.celebrate.lines.01']}, 'rosie-027bafea5b4e76a7': {'assetId': 'rosie-027bafea5b4e76a7', 'role': 'rosie', 'voiceId': 'qwen_rosie_r2', 'text': 'You did it! You got a Princess sticker!', 'filename': 'rosie__rosie-027bafea5b4e76a7__qwen_rosie_r2.wav', 'src': 'audio/rosie__rosie-027bafea5b4e76a7__qwen_rosie_r2.wav', 'lineIds': ['hw.library.reward.lines.01']}, 'princess-2d6b59558a18e5e2': {'assetId': 'princess-2d6b59558a18e5e2', 'role': 'princess', 'voiceId': 'af_bella', 'text': 'Bye-bye, Stephanie!', 'filename': 'princess__princess-2d6b59558a18e5e2__af_bella.wav', 'src': 'audio/princess__princess-2d6b59558a18e5e2__af_bella.wav', 'lineIds': ['hw.library.goodbye.lines.01']}}
PROTECTED = {'rosie-511b9ae26efed5a8': 'efd8173f8f95889f014ea06cf37a6b2fc24e4903fe1f4c735060bf1c5f9af7a1', 'rosie-90eb7bf71f773ef6': 'fa9e972f3a41cd821e9b165c9ac979440031baabca9077de2af9bd0c9879f434', 'rosie-87319e5d959e19f2': 'e5e5d6da05a9b157243f6ce45843956e6cd26e85b76de4c1e4058ca7512e79a8', 'princess-3720e4220c8a5ce4': '75d0d1c5cde1e30dd8292b459307223c629db288cf1040197266774babc00451', 'princess-e728f8e179e85c02': '977bc8e3b6081f9587b24ed26ebdbfa57e27b1f13a58c027e3cd6dd684822461', 'princess-571bf50fb946e5c5': 'baa5c10f30b465afeb9dd63307beb8ce25d59d67e46d994e03e079b10588a4d5'}
EXPECTED_LINES = {'hw.library.welcome.lines.01': {'lineId': 'hw.library.welcome.lines.01', 'assetId': 'rosie-511b9ae26efed5a8', 'role': 'rosie', 'text': 'Welcome to the Halloween Library!', 'scene': 'library', 'nodeId': 'welcome', 'channel': 'lines', 'slot': 1, 'stage': 1, 'view': 'library', 'kind': 'story', 'emotionTarget': 'warm', 'counts': [2, 3, 4, 5], 'origin': 'engine.js', 'motion': None, 'effect': None, 'events': [{'trigger': 'voice.start', 'type': 'mouth.start', 'actor': 'rosie'}, {'trigger': 'voice.end', 'type': 'mouth.stop', 'actor': 'rosie'}], 'actionTiming': None}, 'hw.library.teach-crown.lines.01': {'lineId': 'hw.library.teach-crown.lines.01', 'assetId': 'rosie-90eb7bf71f773ef6', 'role': 'rosie', 'text': 'A crown. Crown.', 'scene': 'library', 'nodeId': 'teach-crown', 'channel': 'lines', 'slot': 1, 'stage': 1, 'view': 'crown', 'kind': 'story', 'emotionTarget': 'warm', 'counts': [2, 3, 4, 5], 'origin': 'engine.js', 'motion': None, 'effect': None, 'events': [{'trigger': 'voice.start', 'type': 'mouth.start', 'actor': 'rosie'}, {'trigger': 'voice.end', 'type': 'mouth.stop', 'actor': 'rosie'}], 'actionTiming': None}, 'hw.library.find-princess.lines.01': {'lineId': 'hw.library.find-princess.lines.01', 'assetId': 'rosie-87319e5d959e19f2', 'role': 'rosie', 'text': 'A crown. A pink dress. Who is it?', 'scene': 'library', 'nodeId': 'find-princess', 'channel': 'lines', 'slot': 1, 'stage': 1, 'view': 'guest-choices', 'kind': 'question', 'emotionTarget': 'curious', 'counts': [2, 3, 4, 5], 'origin': 'engine.js', 'motion': None, 'effect': None, 'events': [{'trigger': 'voice.start', 'type': 'mouth.start', 'actor': 'rosie'}, {'trigger': 'voice.end', 'type': 'mouth.stop', 'actor': 'rosie'}], 'actionTiming': None}, 'hw.library.find-princess.hint.01': {'lineId': 'hw.library.find-princess.hint.01', 'assetId': 'rosie-c190ddf93ee3874a', 'role': 'rosie', 'text': 'Crown. Pink dress.', 'scene': 'library', 'nodeId': 'find-princess', 'channel': 'hint', 'slot': 1, 'stage': 1, 'view': 'guest-choices', 'kind': 'question', 'emotionTarget': 'clear-encouraging', 'counts': [2, 3, 4, 5], 'origin': 'engine.js', 'motion': None, 'effect': None, 'events': [{'trigger': 'voice.start', 'type': 'mouth.start', 'actor': 'rosie'}, {'trigger': 'voice.end', 'type': 'mouth.stop', 'actor': 'rosie'}], 'actionTiming': None}, 'hw.library.find-princess.retry.01': {'lineId': 'hw.library.find-princess.retry.01', 'assetId': 'rosie-7affab230c87e815', 'role': 'rosie', 'text': 'Good try. Look for the pink dress.', 'scene': 'library', 'nodeId': 'find-princess', 'channel': 'retry', 'slot': 1, 'stage': 1, 'view': 'guest-choices', 'kind': 'question', 'emotionTarget': 'gentle-reassuring', 'counts': [2, 3, 4, 5], 'origin': 'engine.js', 'motion': None, 'effect': None, 'events': [{'trigger': 'voice.start', 'type': 'mouth.start', 'actor': 'rosie'}, {'trigger': 'voice.end', 'type': 'mouth.stop', 'actor': 'rosie'}], 'actionTiming': None}, 'hw.library.crown-on.lines.01': {'lineId': 'hw.library.crown-on.lines.01', 'assetId': 'rosie-ec2ecfd3d2ddfdeb', 'role': 'rosie', 'text': 'Yes! She is the princess! The princess has a crown!', 'scene': 'library', 'nodeId': 'crown-on', 'channel': 'lines', 'slot': 1, 'stage': 1, 'view': 'library', 'kind': 'action', 'emotionTarget': 'warm', 'counts': [2, 3, 4, 5], 'origin': 'engine.js', 'motion': 'dance', 'effect': 'crown', 'events': [{'trigger': 'voice.start', 'type': 'mouth.start', 'actor': 'rosie'}, {'trigger': 'voice.end', 'type': 'mouth.stop', 'actor': 'rosie'}], 'actionTiming': 'Preserve existing action phase after dialogue; use playback clock, never infer action timing from RMS.'}, 'hw.library.hello.lines.01': {'lineId': 'hw.library.hello.lines.01', 'assetId': 'princess-3720e4220c8a5ce4', 'role': 'princess', 'text': 'Hello, Stephanie!', 'scene': 'library', 'nodeId': 'hello', 'channel': 'lines', 'slot': 1, 'stage': 1, 'view': 'library', 'kind': 'natural', 'emotionTarget': 'warm-friendly', 'counts': [2, 3, 4, 5], 'origin': 'engine.js', 'motion': None, 'effect': None, 'events': [{'trigger': 'voice.start', 'type': 'mouth.start', 'actor': 'princess'}, {'trigger': 'voice.end', 'type': 'mouth.stop', 'actor': 'princess'}, {'trigger': 'voice.start', 'type': 'existing-greeting-gesture', 'actor': 'princess'}], 'actionTiming': None}, 'hw.library.find-book.lines.01': {'lineId': 'hw.library.find-book.lines.01', 'assetId': 'princess-e728f8e179e85c02', 'role': 'princess', 'text': 'Where is my book? My book has a crown.', 'scene': 'library', 'nodeId': 'find-book', 'channel': 'lines', 'slot': 1, 'stage': 2, 'view': 'book-choices', 'kind': 'question', 'emotionTarget': 'curious', 'counts': [2, 3, 4, 5], 'origin': 'engine.js', 'motion': None, 'effect': None, 'events': [{'trigger': 'voice.start', 'type': 'mouth.start', 'actor': 'princess'}, {'trigger': 'voice.end', 'type': 'mouth.stop', 'actor': 'princess'}], 'actionTiming': None}, 'hw.library.find-book.lines.02': {'lineId': 'hw.library.find-book.lines.02', 'assetId': 'rosie-68054a70c21c65b6', 'role': 'rosie', 'text': 'Find the crown book. Which one?', 'scene': 'library', 'nodeId': 'find-book', 'channel': 'lines', 'slot': 2, 'stage': 2, 'view': 'book-choices', 'kind': 'question', 'emotionTarget': 'curious', 'counts': [2, 3, 4, 5], 'origin': 'engine.js', 'motion': None, 'effect': None, 'events': [{'trigger': 'voice.start', 'type': 'mouth.start', 'actor': 'rosie'}, {'trigger': 'voice.end', 'type': 'mouth.stop', 'actor': 'rosie'}], 'actionTiming': None}, 'hw.library.find-book.hint.01': {'lineId': 'hw.library.find-book.hint.01', 'assetId': 'rosie-d1b47e593b3a21df', 'role': 'rosie', 'text': 'Crown. Find the crown book.', 'scene': 'library', 'nodeId': 'find-book', 'channel': 'hint', 'slot': 1, 'stage': 2, 'view': 'book-choices', 'kind': 'question', 'emotionTarget': 'clear-encouraging', 'counts': [2, 3, 4, 5], 'origin': 'engine.js', 'motion': None, 'effect': None, 'events': [{'trigger': 'voice.start', 'type': 'mouth.start', 'actor': 'rosie'}, {'trigger': 'voice.end', 'type': 'mouth.stop', 'actor': 'rosie'}], 'actionTiming': None}, 'hw.library.find-book.retry.01': {'lineId': 'hw.library.find-book.retry.01', 'assetId': 'rosie-b533d7376f1b7f96', 'role': 'rosie', 'text': 'Good try. Look for the crown.', 'scene': 'library', 'nodeId': 'find-book', 'channel': 'retry', 'slot': 1, 'stage': 2, 'view': 'book-choices', 'kind': 'question', 'emotionTarget': 'gentle-reassuring', 'counts': [2, 3, 4, 5], 'origin': 'engine.js', 'motion': None, 'effect': None, 'events': [{'trigger': 'voice.start', 'type': 'mouth.start', 'actor': 'rosie'}, {'trigger': 'voice.end', 'type': 'mouth.stop', 'actor': 'rosie'}], 'actionTiming': None}, 'hw.library.book-found.lines.01': {'lineId': 'hw.library.book-found.lines.01', 'assetId': 'rosie-6bfa133bd1d20605', 'role': 'rosie', 'text': 'Yes! The crown book!', 'scene': 'library', 'nodeId': 'book-found', 'channel': 'lines', 'slot': 1, 'stage': 2, 'view': 'library', 'kind': 'action', 'emotionTarget': 'happy', 'counts': [2, 3, 4, 5], 'origin': 'engine.js', 'motion': 'deliver', 'effect': 'book', 'events': [{'trigger': 'voice.start', 'type': 'mouth.start', 'actor': 'rosie'}, {'trigger': 'voice.end', 'type': 'mouth.stop', 'actor': 'rosie'}], 'actionTiming': 'Preserve existing action phase after dialogue; use playback clock, never infer action timing from RMS.'}, 'hw.library.book-found.lines.02': {'lineId': 'hw.library.book-found.lines.02', 'assetId': 'princess-571bf50fb946e5c5', 'role': 'princess', 'text': 'I found my book! Thank you, Stephanie!', 'scene': 'library', 'nodeId': 'book-found', 'channel': 'lines', 'slot': 2, 'stage': 2, 'view': 'library', 'kind': 'action', 'emotionTarget': 'happy', 'counts': [2, 3, 4, 5], 'origin': 'engine.js', 'motion': 'deliver', 'effect': 'book', 'events': [{'trigger': 'voice.start', 'type': 'mouth.start', 'actor': 'princess'}, {'trigger': 'voice.end', 'type': 'mouth.stop', 'actor': 'princess'}], 'actionTiming': 'Preserve existing action phase after dialogue; use playback clock, never infer action timing from RMS.'}, 'hw.library.open-book.lines.01': {'lineId': 'hw.library.open-book.lines.01', 'assetId': 'princess-666130f81a2643cb', 'role': 'princess', 'text': 'Let’s open the book!', 'scene': 'library', 'nodeId': 'open-book', 'channel': 'lines', 'slot': 1, 'stage': 3, 'view': 'library', 'kind': 'action', 'emotionTarget': 'warm', 'counts': [2, 3, 4, 5], 'origin': 'engine.js', 'motion': 'open', 'effect': 'open', 'events': [{'trigger': 'voice.start', 'type': 'mouth.start', 'actor': 'princess'}, {'trigger': 'voice.end', 'type': 'mouth.stop', 'actor': 'princess'}], 'actionTiming': 'Preserve existing action phase after dialogue; use playback clock, never infer action timing from RMS.'}, 'hw.library.stephanie-says.lines.01': {'lineId': 'hw.library.stephanie-says.lines.01', 'assetId': 'princess-00ce7ab0e0fa0e49', 'role': 'princess', 'text': 'This is a magic book! Let’s play Stephanie Says!', 'scene': 'library', 'nodeId': 'stephanie-says', 'channel': 'lines', 'slot': 1, 'stage': 3, 'view': 'library', 'kind': 'story', 'emotionTarget': 'warm', 'counts': [2, 3, 4, 5], 'origin': 'engine.js', 'motion': None, 'effect': None, 'events': [{'trigger': 'voice.start', 'type': 'mouth.start', 'actor': 'princess'}, {'trigger': 'voice.end', 'type': 'mouth.stop', 'actor': 'princess'}], 'actionTiming': None}, 'hw.library.stephanie-says.lines.02': {'lineId': 'hw.library.stephanie-says.lines.02', 'assetId': 'princess-81eaaa24e2930533', 'role': 'princess', 'text': 'You’re the teacher, Stephanie. I’ll follow you!', 'scene': 'library', 'nodeId': 'stephanie-says', 'channel': 'lines', 'slot': 2, 'stage': 3, 'view': 'library', 'kind': 'story', 'emotionTarget': 'warm', 'counts': [2, 3, 4, 5], 'origin': 'engine.js', 'motion': None, 'effect': None, 'events': [{'trigger': 'voice.start', 'type': 'mouth.start', 'actor': 'princess'}, {'trigger': 'voice.end', 'type': 'mouth.stop', 'actor': 'princess'}], 'actionTiming': None}, 'hw.library.command-1.lines.01': {'lineId': 'hw.library.command-1.lines.01', 'assetId': 'princess-f62fe81d9ea55595', 'role': 'princess', 'text': 'I’m ready, Teacher Stephanie!', 'scene': 'library', 'nodeId': 'command-1', 'channel': 'lines', 'slot': 1, 'stage': 3, 'view': 'library', 'kind': 'question', 'emotionTarget': 'curious', 'counts': [2, 3, 4, 5], 'origin': 'engine.js', 'motion': None, 'effect': None, 'events': [{'trigger': 'voice.start', 'type': 'mouth.start', 'actor': 'princess'}, {'trigger': 'voice.end', 'type': 'mouth.stop', 'actor': 'princess'}], 'actionTiming': None}, 'hw.library.command-1.lines.02': {'lineId': 'hw.library.command-1.lines.02', 'assetId': 'princess-f4203d98fce906d9', 'role': 'princess', 'text': 'Turn around, wave your hands, climb, or rollerblade?', 'scene': 'library', 'nodeId': 'command-1', 'channel': 'lines', 'slot': 2, 'stage': 3, 'view': 'library', 'kind': 'question', 'emotionTarget': 'curious', 'counts': [2, 3, 4, 5], 'origin': 'engine.js', 'motion': None, 'effect': None, 'events': [{'trigger': 'voice.start', 'type': 'mouth.start', 'actor': 'princess'}, {'trigger': 'voice.end', 'type': 'mouth.stop', 'actor': 'princess'}], 'actionTiming': None}, 'hw.library.command-1.lines.03': {'lineId': 'hw.library.command-1.lines.03', 'assetId': 'rosie-9c39282e9c49070c', 'role': 'rosie', 'text': 'Choose one. Tell me: “Stephanie says…”', 'scene': 'library', 'nodeId': 'command-1', 'channel': 'lines', 'slot': 3, 'stage': 3, 'view': 'library', 'kind': 'question', 'emotionTarget': 'curious', 'counts': [2, 3, 4, 5], 'origin': 'engine.js', 'motion': None, 'effect': None, 'events': [{'trigger': 'voice.start', 'type': 'mouth.start', 'actor': 'rosie'}, {'trigger': 'voice.end', 'type': 'mouth.stop', 'actor': 'rosie'}], 'actionTiming': None}, 'hw.library.command-1.hint.01': {'lineId': 'hw.library.command-1.hint.01', 'assetId': 'rosie-2518646d9595742a', 'role': 'rosie', 'text': 'Say: “Stephanie says, wave your hands!”', 'scene': 'library', 'nodeId': 'command-1', 'channel': 'hint', 'slot': 1, 'stage': 3, 'view': 'library', 'kind': 'question', 'emotionTarget': 'clear-encouraging', 'counts': [2, 3, 4, 5], 'origin': 'engine.js', 'motion': None, 'effect': None, 'events': [{'trigger': 'voice.start', 'type': 'mouth.start', 'actor': 'rosie'}, {'trigger': 'voice.end', 'type': 'mouth.stop', 'actor': 'rosie'}], 'actionTiming': None}, 'hw.library.move-1.lines.01': {'lineId': 'hw.library.move-1.lines.01', 'assetId': 'princess-02986f8d4e57964b', 'role': 'princess', 'text': 'Stephanie says… I’ll do it!', 'scene': 'library', 'nodeId': 'move-1', 'channel': 'lines', 'slot': 1, 'stage': 3, 'view': 'library', 'kind': 'action', 'emotionTarget': 'warm', 'counts': [2, 3, 4, 5], 'origin': 'engine.js', 'motion': 'chosen', 'effect': 'princess-move', 'events': [{'trigger': 'voice.start', 'type': 'mouth.start', 'actor': 'princess'}, {'trigger': 'voice.end', 'type': 'mouth.stop', 'actor': 'princess'}], 'actionTiming': 'Preserve existing action phase after dialogue; use playback clock, never infer action timing from RMS.'}, 'hw.library.command-2.lines.01': {'lineId': 'hw.library.command-2.lines.01', 'assetId': 'princess-f49de3f6f81d17d7', 'role': 'princess', 'text': 'Your turn again, Stephanie! What should I do?', 'scene': 'library', 'nodeId': 'command-2', 'channel': 'lines', 'slot': 1, 'stage': 3, 'view': 'library', 'kind': 'question', 'emotionTarget': 'curious', 'counts': [2, 3, 4, 5], 'origin': 'engine.js', 'motion': None, 'effect': None, 'events': [{'trigger': 'voice.start', 'type': 'mouth.start', 'actor': 'princess'}, {'trigger': 'voice.end', 'type': 'mouth.stop', 'actor': 'princess'}], 'actionTiming': None}, 'hw.library.command-2.lines.02': {'lineId': 'hw.library.command-2.lines.02', 'assetId': 'princess-f4203d98fce906d9', 'role': 'princess', 'text': 'Turn around, wave your hands, climb, or rollerblade?', 'scene': 'library', 'nodeId': 'command-2', 'channel': 'lines', 'slot': 2, 'stage': 3, 'view': 'library', 'kind': 'question', 'emotionTarget': 'curious', 'counts': [2, 3, 4, 5], 'origin': 'engine.js', 'motion': None, 'effect': None, 'events': [{'trigger': 'voice.start', 'type': 'mouth.start', 'actor': 'princess'}, {'trigger': 'voice.end', 'type': 'mouth.stop', 'actor': 'princess'}], 'actionTiming': None}, 'hw.library.command-2.lines.03': {'lineId': 'hw.library.command-2.lines.03', 'assetId': 'rosie-9c39282e9c49070c', 'role': 'rosie', 'text': 'Choose one. Tell me: “Stephanie says…”', 'scene': 'library', 'nodeId': 'command-2', 'channel': 'lines', 'slot': 3, 'stage': 3, 'view': 'library', 'kind': 'question', 'emotionTarget': 'curious', 'counts': [2, 3, 4, 5], 'origin': 'engine.js', 'motion': None, 'effect': None, 'events': [{'trigger': 'voice.start', 'type': 'mouth.start', 'actor': 'rosie'}, {'trigger': 'voice.end', 'type': 'mouth.stop', 'actor': 'rosie'}], 'actionTiming': None}, 'hw.library.command-2.hint.01': {'lineId': 'hw.library.command-2.hint.01', 'assetId': 'rosie-2518646d9595742a', 'role': 'rosie', 'text': 'Say: “Stephanie says, wave your hands!”', 'scene': 'library', 'nodeId': 'command-2', 'channel': 'hint', 'slot': 1, 'stage': 3, 'view': 'library', 'kind': 'question', 'emotionTarget': 'clear-encouraging', 'counts': [2, 3, 4, 5], 'origin': 'engine.js', 'motion': None, 'effect': None, 'events': [{'trigger': 'voice.start', 'type': 'mouth.start', 'actor': 'rosie'}, {'trigger': 'voice.end', 'type': 'mouth.stop', 'actor': 'rosie'}], 'actionTiming': None}, 'hw.library.move-2.lines.01': {'lineId': 'hw.library.move-2.lines.01', 'assetId': 'princess-02986f8d4e57964b', 'role': 'princess', 'text': 'Stephanie says… I’ll do it!', 'scene': 'library', 'nodeId': 'move-2', 'channel': 'lines', 'slot': 1, 'stage': 3, 'view': 'library', 'kind': 'action', 'emotionTarget': 'warm', 'counts': [2, 3, 4, 5], 'origin': 'engine.js', 'motion': 'chosen', 'effect': 'princess-move', 'events': [{'trigger': 'voice.start', 'type': 'mouth.start', 'actor': 'princess'}, {'trigger': 'voice.end', 'type': 'mouth.stop', 'actor': 'princess'}], 'actionTiming': 'Preserve existing action phase after dialogue; use playback clock, never infer action timing from RMS.'}, 'hw.library.pages.lines.01': {'lineId': 'hw.library.pages.lines.01', 'assetId': 'rosie-4682559e0c393296', 'role': 'rosie', 'text': 'You’re a great teacher, Stephanie!', 'scene': 'library', 'nodeId': 'pages', 'channel': 'lines', 'slot': 1, 'stage': 3, 'view': 'pages', 'kind': 'story', 'emotionTarget': 'warm', 'counts': [2, 3, 4, 5], 'origin': 'engine.js', 'motion': None, 'effect': None, 'events': [{'trigger': 'voice.start', 'type': 'mouth.start', 'actor': 'rosie'}, {'trigger': 'voice.end', 'type': 'mouth.stop', 'actor': 'rosie'}], 'actionTiming': None}, 'hw.library.pages.lines.02': {'lineId': 'hw.library.pages.lines.02', 'assetId': 'princess-9df3ec60229e839e', 'role': 'princess', 'text': 'I loved playing Stephanie Says with you!', 'scene': 'library', 'nodeId': 'pages', 'channel': 'lines', 'slot': 2, 'stage': 3, 'view': 'pages', 'kind': 'story', 'emotionTarget': 'warm', 'counts': [2, 3, 4, 5], 'origin': 'engine.js', 'motion': None, 'effect': None, 'events': [{'trigger': 'voice.start', 'type': 'mouth.start', 'actor': 'princess'}, {'trigger': 'voice.end', 'type': 'mouth.stop', 'actor': 'princess'}], 'actionTiming': None}, 'hw.library.close-book.lines.01': {'lineId': 'hw.library.close-book.lines.01', 'assetId': 'rosie-f4f915314162a16c', 'role': 'rosie', 'text': 'Great job, Stephanie!', 'scene': 'library', 'nodeId': 'close-book', 'channel': 'lines', 'slot': 1, 'stage': 4, 'view': 'library', 'kind': 'action', 'emotionTarget': 'warm', 'counts': [2, 3, 4, 5], 'origin': 'engine.js', 'motion': 'close', 'effect': 'close', 'events': [{'trigger': 'voice.start', 'type': 'mouth.start', 'actor': 'rosie'}, {'trigger': 'voice.end', 'type': 'mouth.stop', 'actor': 'rosie'}], 'actionTiming': 'Preserve existing action phase after dialogue; use playback clock, never infer action timing from RMS.'}, 'hw.library.happy-question.lines.01': {'lineId': 'hw.library.happy-question.lines.01', 'assetId': 'princess-439256ed3b9e572a', 'role': 'princess', 'text': 'I’m happy!', 'scene': 'library', 'nodeId': 'happy-question', 'channel': 'lines', 'slot': 1, 'stage': 4, 'view': 'library', 'kind': 'natural', 'emotionTarget': 'warm', 'counts': [2, 3, 4, 5], 'origin': 'engine.js', 'motion': None, 'effect': None, 'events': [{'trigger': 'voice.start', 'type': 'mouth.start', 'actor': 'princess'}, {'trigger': 'voice.end', 'type': 'mouth.stop', 'actor': 'princess'}], 'actionTiming': None}, 'hw.library.happy-question.lines.02': {'lineId': 'hw.library.happy-question.lines.02', 'assetId': 'rosie-440a07b589744f63', 'role': 'rosie', 'text': 'Are you happy?', 'scene': 'library', 'nodeId': 'happy-question', 'channel': 'lines', 'slot': 2, 'stage': 4, 'view': 'library', 'kind': 'natural', 'emotionTarget': 'warm', 'counts': [2, 3, 4, 5], 'origin': 'engine.js', 'motion': None, 'effect': None, 'events': [{'trigger': 'voice.start', 'type': 'mouth.start', 'actor': 'rosie'}, {'trigger': 'voice.end', 'type': 'mouth.stop', 'actor': 'rosie'}], 'actionTiming': None}, 'hw.library.celebrate.lines.01': {'lineId': 'hw.library.celebrate.lines.01', 'assetId': 'princess-cde0b237677f051f', 'role': 'princess', 'text': 'Let’s dance!', 'scene': 'library', 'nodeId': 'celebrate', 'channel': 'lines', 'slot': 1, 'stage': 4, 'view': 'library', 'kind': 'action', 'emotionTarget': 'warm', 'counts': [2, 3, 4, 5], 'origin': 'engine.js', 'motion': 'celebrate', 'effect': None, 'events': [{'trigger': 'voice.start', 'type': 'mouth.start', 'actor': 'princess'}, {'trigger': 'voice.end', 'type': 'mouth.stop', 'actor': 'princess'}], 'actionTiming': 'Preserve existing action phase after dialogue; use playback clock, never infer action timing from RMS.'}, 'hw.library.reward.lines.01': {'lineId': 'hw.library.reward.lines.01', 'assetId': 'rosie-027bafea5b4e76a7', 'role': 'rosie', 'text': 'You did it! You got a Princess sticker!', 'scene': 'library', 'nodeId': 'reward', 'channel': 'lines', 'slot': 1, 'stage': 4, 'view': 'reward', 'kind': 'action', 'emotionTarget': 'happy', 'counts': [2, 3, 4, 5], 'origin': 'engine.js', 'motion': 'reward', 'effect': 'reward', 'events': [{'trigger': 'voice.start', 'type': 'mouth.start', 'actor': 'rosie'}, {'trigger': 'voice.end', 'type': 'mouth.stop', 'actor': 'rosie'}], 'actionTiming': 'Preserve existing action phase after dialogue; use playback clock, never infer action timing from RMS.'}, 'hw.library.goodbye.lines.01': {'lineId': 'hw.library.goodbye.lines.01', 'assetId': 'princess-2d6b59558a18e5e2', 'role': 'princess', 'text': 'Bye-bye, Stephanie!', 'scene': 'library', 'nodeId': 'goodbye', 'channel': 'lines', 'slot': 1, 'stage': 4, 'view': 'reward', 'kind': 'natural', 'emotionTarget': 'warm-friendly', 'counts': [2, 3, 4, 5], 'origin': 'engine.js', 'motion': None, 'effect': None, 'events': [{'trigger': 'voice.start', 'type': 'mouth.start', 'actor': 'princess'}, {'trigger': 'voice.end', 'type': 'mouth.stop', 'actor': 'princess'}], 'actionTiming': None}}

def digest(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def read_json(p): return json.loads(Path(p).read_text(encoding='utf-8-sig'))
def write_json(p,data):
 p=Path(p);tmp=p.with_suffix('.tmp');tmp.write_text(json.dumps(data,ensure_ascii=False,indent=2),encoding='utf-8');tmp.replace(p)
def checked_path(out,src):
 p=(out/src).resolve()
 if not p.is_relative_to(out.resolve()):raise ValueError('音檔路徑不符')
 return p

def validate(out):
 out=Path(out).resolve();mf=out/'generated-manifest.json';m=read_json(mf)
 if m.get('scope')!='library-complete30-review' or set(m['assets'])!=set(EXPECTED):raise ValueError('需要完整Library 30句manifest，請確認OUTPUT')
 if m['lines']!=EXPECTED_LINES:raise ValueError('lineId、字幕或事件對應已改變')
 for aid,e in EXPECTED.items():
  a=m['assets'][aid]
  if any(a.get(k)!=v for k,v in e.items()):raise ValueError('角色、聲線、台詞或assetId不符：'+aid)
  if a.get('lineId') not in e['lineIds']:raise ValueError('固定lineId不符：'+aid)
  p=checked_path(out,a['src']);h=digest(p)
  if h!=a['sha256'] or (aid in PROTECTED and h!=PROTECTED[aid]):raise ValueError('WAV SHA256不符：'+aid)
  data,sr=sf.read(p)
  if not np.isfinite(data).all() or not .12<len(data)/sr<45 or abs(len(data)/sr-a['duration'])>.03:raise ValueError('音訊解碼/長度異常：'+aid)
 return m,digest(mf)

def identity(a):return {k:a[k] for k in ('assetId','lineId','role','voiceId','text','sha256')}
def new_review(m,mh):
 return {'schemaVersion':1,'scope':'library30-human-review','manifestSha256':mh,'productionApproved':False,
  'lines':{aid:{**identity(a),'decision':'accepted' if aid in PROTECTED else 'pending','notes':'','locked':aid in PROTECTED} for aid,a in m['assets'].items()}}
def validate_review(review,m,mh):
 if review.get('scope')!='library30-human-review' or review.get('manifestSha256')!=mh or set(review.get('lines',{}))!=set(EXPECTED):raise ValueError('驗收紀錄不屬於目前這30句音檔')
 for aid,a in m['assets'].items():
  row=review['lines'][aid]
  if any(row.get(k)!=v for k,v in identity(a).items()):raise ValueError('驗收紀錄與音檔不符：'+aid)
  if row.get('decision') not in ('accepted','needs-retake','pending') or not isinstance(row.get('notes'),str):raise ValueError('驗收選項不符')
  if aid in PROTECTED and (row['decision']!='accepted' or row.get('locked') is not True or row['notes']!=''):raise ValueError('已接受六句不可修改')
 return review

def persist(out,review,m,mh):
 validate_review(review,m,mh)
 review['updatedAt']=datetime.datetime.now(datetime.timezone.utc).isoformat()
 write_json(Path(out)/'voice-review.json',review)
 retakes={'scope':'library-specified-retake-request','generationEnabled':False,'productionApproved':False,
  'manifestSha256':mh,'jobs':[r for aid,r in review['lines'].items() if aid not in PROTECTED and r['decision']=='needs-retake']}
 write_json(Path(out)/'retake-request.json',retakes)
 return retakes

def export_bundle(out,review,m,mh):
 current,now=validate(out)
 if now!=mh:raise ValueError('manifest已變動，停止匯出')
 persist(out,review,m,mh)
 dest=Path(out).parent/('library30-reviewed-'+uuid.uuid4().hex[:8]+'.zip')
 with zipfile.ZipFile(dest,'w',zipfile.ZIP_DEFLATED) as z:
  for name in ('generated-manifest.json','voice-review.json','retake-request.json'):z.write(Path(out)/name,name)
  for a in m['assets'].values():z.write(checked_path(Path(out),a['src']),a['src'])
  cards=[]
  for aid,a in m['assets'].items():
   row=review['lines'][aid];label={'accepted':'已接受','needs-retake':'需要重錄','pending':'尚未決定'}[row['decision']]
   cards.append('<section><h3>'+html.escape(a['role']+' / '+a['voiceId'])+'</h3><p>'+html.escape(a['lineId'])+'</p><p>'+html.escape(a['text'])+'</p><audio controls preload="none" src="'+html.escape(a['src'],quote=True)+'"></audio><p>'+label+('（鎖定）' if aid in PROTECTED else '')+'</p><p>'+html.escape(row['notes'])+'</p></section>')
  z.writestr('review.html','<!doctype html><meta charset="utf-8"><title>Library驗收紀錄</title><style>body{max-width:850px;margin:auto;font:17px/1.6 sans-serif}section{padding:16px;border-bottom:1px solid #aaa}</style><h1>Library 30句驗收紀錄</h1>'+''.join(cards))
 return dest

def show_review(output):
 from IPython.display import Audio,display,clear_output
 import ipywidgets as widgets
 from google.colab import files
 out=Path(output).resolve();m,mh=validate(out);rp=out/'voice-review.json'
 review=validate_review(read_json(rp),m,mh) if rp.exists() else new_review(m,mh)
 persist(out,review,m,mh)
 status=widgets.HTML();controls={};loading={'active':False};token=uuid.uuid4().hex;globals()['_library_review_token']=token
 def summary(message=''):
  rows=list(review['lines'].values());status.value=html.escape(message)+'<p>已接受 '+str(sum(x['decision']=='accepted' for x in rows))+' / 30（含鎖定6句）；需要重錄 '+str(sum(x['decision']=='needs-retake' for x in rows))+'；尚未決定 '+str(sum(x['decision']=='pending' for x in rows))+'。選擇與備註會自動保存至 OUTPUT/voice-review.json。</p>'
 def changed(aid,choice,notes):
  def on_change(_):
   if loading['active'] or globals().get('_library_review_token')!=token:return
   review['lines'][aid].update(decision=choice.value,notes=notes.value)
   try:persist(out,review,m,mh);summary('已自動保存')
   except Exception as e:summary('保存失敗：'+str(e))
  return on_change
 download=widgets.Button(description='下載音檔與驗收紀錄',layout=widgets.Layout(width='220px'))
 export=widgets.Button(description='匯出驗收紀錄 JSON',layout=widgets.Layout(width='210px'))
 reload_button=widgets.Button(description='重新載入驗收紀錄 JSON',layout=widgets.Layout(width='230px'))
 actions=widgets.Output()
 def save_all(_):
  with actions:
   clear_output(wait=True)
   try:files.download(str(export_bundle(out,review,m,mh)));summary('已核對30句並建立完整ZIP')
   except Exception as e:print('匯出失敗：',e)
 def save_review(_):
  persist(out,review,m,mh);files.download(str(rp))
 def reload_review(_):
  nonlocal review
  with actions:
   clear_output(wait=True)
   try:
    uploaded=files.upload()
    if len(uploaded)!=1:raise ValueError('請只選一個 voice-review.json；不需要上傳WAV')
    incoming=validate_review(json.loads(next(iter(uploaded.values())).decode('utf-8-sig')),m,mh)
    # Invalid imports never replace current choices. A valid full record replaces all24 editable decisions.
    if rp.exists():rp.with_name('voice-review-backup-'+uuid.uuid4().hex[:8]+'.json').write_bytes(rp.read_bytes())
    review=incoming;loading['active']=True
    for aid,(choice,notes) in controls.items():choice.value=review['lines'][aid]['decision'];notes.value=review['lines'][aid]['notes']
    # Widget observers can write while restoring; persist the complete imported record again.
    loading['active']=False
    persist(out,review,m,mh);summary('已載入驗收紀錄，載入前紀錄另存備份')
   except Exception as e:
    loading['active']=False;print('載入失敗：',e)
 download.on_click(save_all);export.on_click(save_review);reload_button.on_click(reload_review)
 display(widgets.HTML('<h2>Library 30句逐句人工驗收</h2><p>已核對30個WAV SHA256與固定台詞；已接受六句鎖定。本區塊不會生成任何聲音。Colab執行階段重啟會清除暫存檔，離開前請下載ZIP或驗收JSON。</p>'),widgets.HBox([download,export,reload_button]),status,actions)
 for aid,a in m['assets'].items():
  display(widgets.HTML('<hr><h3>'+html.escape(a['role']+' / '+a['voiceId'])+'</h3><p>'+html.escape(a['text'])+'</p><small>'+html.escape(a['lineId']+' · '+aid)+'</small>'))
  display(Audio(filename=str(checked_path(out,a['src'])),autoplay=False))
  if aid in PROTECTED:display(widgets.HTML('<p><b>已接受（原六句鎖定，不可修改）</b></p>'));continue
  row=review['lines'][aid]
  choice=widgets.Dropdown(options=[('尚未決定','pending'),('接受此句','accepted'),('需要重錄','needs-retake')],value=row['decision'],description='人工驗收：')
  notes=widgets.Textarea(value=row['notes'],description='備註：',placeholder='音色太成熟、語速太快、漏詞、發音問題等',continuous_update=True,layout=widgets.Layout(width='95%',height='75px'))
  callback=changed(aid,choice,notes);choice.observe(callback,names='value');notes.observe(callback,names='value');controls[aid]=(choice,notes);display(choice,notes)
 summary('已載入先前驗收紀錄' if rp.exists() else '驗收介面已準備')
 return {'manifest':m,'review':review,'controls':controls,'output':out}

if 'OUTPUT' in globals():
 LIBRARY_REVIEW_UI=show_review(OUTPUT)


## 驗收與授權
已接受六句維持原WAV與SHA256。新24句仍需人工檢查童真、溫柔、漏詞及聲線一致性，未自動核准。Qwen/Kokoro Apache-2.0；參考音僅依家長核准用途，權利來源未另行獨立稽核，勿公開參考包。mouth events為台詞層級及音量動畫，不是精準音素嘴型。